# PRO_CHECKS — Frontier model full solutions for verifier-rejected rows

Reads a verifier-rejected CSV (currently JEE maths eval rejects) and processes **only** rows where:
- `status = NO_ANSWER`
- `sub_status = VERIFIER_REJECTED`

SUCCESS rows are **not** sent to Gemini.

Writes **only** `FRONTIER_MODEL_RESPONSE` (full Gemini solution) back to the same CSV.
Input validation still runs internally before each API call.

Images are downloaded locally and sent as inline bytes (not `from_uri`).
Uses parallel workers (`MAX_WORKERS`) to speed up Gemini calls.
API key: `/Users/simrannaik/Desktop/bots/.env` (`GOOGLE_GEMINI_API`).

In [1]:
# Install deps if needed
import importlib
import subprocess
import sys

REQUIRED = {
    "dotenv": "python-dotenv",
    "google.genai": "google-genai",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "requests": "requests",
    "PIL": "pillow",
}

for module_name, package_name in REQUIRED.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])

In [3]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from tqdm import tqdm

ENV_PATH = Path("/Users/simrannaik/Desktop/bots/.env")
CSV_PATH = Path("/Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/3_rej_jee_math_truncated/eval_results_unified_145_merged_19_rejected.csv")

# Only process verifier-rejected rows
TARGET_STATUS = "NO_ANSWER"
TARGET_SUB_STATUS = "VERIFIER_REJECTED"

MODEL_NAME = "gemini-3.1-pro-preview"

SAVE_EVERY = 5          # checkpoint interval
SKIP_COMPLETED = True   # skip target rows that already have a response
MAX_WORKERS = 5         # parallel Gemini calls (4-5 is usually safe)
MAX_RETRIES = 3

load_dotenv(ENV_PATH)
api_key = os.getenv("GOOGLE_GEMINI_API") or os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(f"GOOGLE_GEMINI_API not found in {ENV_PATH}")

client = genai.Client(api_key=api_key)
print(f"Model: {MODEL_NAME}")
print(f"CSV: {CSV_PATH}")
print(f"Target filter: status={TARGET_STATUS}, sub_status={TARGET_SUB_STATUS}")

Model: gemini-3.1-pro-preview
CSV: /Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/3_rej_jee_math_truncated/eval_results_unified_145_merged_19_rejected.csv
Target filter: status=NO_ANSWER, sub_status=VERIFIER_REJECTED


In [4]:
import io
import json
import mimetypes
import re
from urllib.parse import urlparse

import requests
from PIL import Image

MIN_IMAGE_BYTES = 500
MIN_OCR_CHARS = 10
MIN_SOLUTION_CHARS = 200

SYSTEM_PROMPT = """You are an expert JEE Mains Maths tutor.

You will receive:
1. Question OCR text
2. Optional additional input text from the student
3. Optional question image

Write a complete, detailed step-by-step solution to the question.

Requirements:
- Solve the full problem from start to finish.
- Explain the approach first, then work through every important step.
- Show all key equations, substitutions, and reasoning clearly.
- For MCQs, evaluate the options when helpful and justify the correct choice.
- If a diagram is present in the image, use it in your reasoning.
- Use clear markdown headings such as ### Introduction, ### Step 1 ### Final Answer.
- Use display math with \\[ ... \\] for equations.
- End with ### Final Answer and state the answer clearly.
"""


def clean_text(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).strip()


def sniff_image_format(image_bytes: bytes) -> str:
    if not image_bytes:
        return "empty"
    if image_bytes.startswith(b"\xff\xd8\xff"):
        return "jpeg"
    if image_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
        return "png"
    if image_bytes.startswith(b"GIF87a") or image_bytes.startswith(b"GIF89a"):
        return "gif"
    if image_bytes.startswith(b"RIFF") and len(image_bytes) > 12 and image_bytes[8:12] == b"WEBP":
        return "webp"
    if image_bytes.startswith(b"<") or image_bytes.startswith(b"<!"):
        return "html"
    return "unknown"


def guess_mime_type(image_url: str, image_bytes: bytes | None = None) -> str:
    sniffed = sniff_image_format(image_bytes or b"")
    mapping = {
        "jpeg": "image/jpeg",
        "png": "image/png",
        "gif": "image/gif",
        "webp": "image/webp",
    }
    if sniffed in mapping:
        return mapping[sniffed]

    path = image_url.lower().split("?")[0]
    mime = mimetypes.guess_type(path)[0]
    if mime and mime.startswith("image/"):
        return mime
    return "image/jpeg"


def is_valid_http_url(url: str) -> bool:
    try:
        parsed = urlparse(url)
        return parsed.scheme in {"http", "https"} and bool(parsed.netloc)
    except Exception:
        return False


def validate_image_bytes(image_bytes: bytes, mime_type: str) -> dict:
    checks = {}
    checks["bytes_nonempty"] = len(image_bytes) > 0
    checks["bytes_min_size"] = len(image_bytes) >= MIN_IMAGE_BYTES
    sniffed = sniff_image_format(image_bytes)
    checks["magic_bytes_valid"] = sniffed in {"jpeg", "png", "gif", "webp"}
    checks["mime_is_image"] = mime_type.startswith("image/")

    pil_ok = False
    pil_info = ""
    try:
        with Image.open(io.BytesIO(image_bytes)) as img:
            img.verify()
        with Image.open(io.BytesIO(image_bytes)) as img2:
            pil_ok = True
            pil_info = f"{img2.format} {img2.size[0]}x{img2.size[1]}"
    except Exception as exc:
        pil_info = str(exc)
    checks["pil_decodable"] = pil_ok

    status = "PASS" if all(checks.values()) else "FAIL"
    notes = {
        "byte_length": len(image_bytes),
        "sniffed_format": sniffed,
        "mime_type": mime_type,
        "pil_info": pil_info,
        "checks": checks,
    }
    return {"status": status, "notes": notes, "image_bytes": image_bytes, "mime_type": mime_type}


def download_image_bytes(image_url: str, timeout: int = 30) -> tuple[bytes, str, int]:
    response = requests.get(
        image_url,
        timeout=timeout,
        headers={"User-Agent": "Mozilla/5.0 (compatible; PRO_CHECKS)"},
    )
    status_code = response.status_code
    response.raise_for_status()
    image_bytes = response.content
    content_type = (response.headers.get("content-type") or "").split(";")[0].strip().lower()
    if not content_type.startswith("image/"):
        content_type = guess_mime_type(image_url, image_bytes)
    return image_bytes, content_type, status_code


def resolve_image_url(row) -> str | None:
    """Pick the first image URL that downloads and decodes as a valid image."""
    candidates = []
    for key in (
        "static_image_url",
        "current_input_image_url",
        "image_url",
        "image",
    ):
        val = clean_text(row.get(key))
        if val and is_valid_http_url(val) and val not in candidates:
            candidates.append(val)

    for url in candidates:
        try:
            image_bytes, mime_type, _ = download_image_bytes(url)
            payload = validate_image_bytes(image_bytes, mime_type)
            if payload["status"] == "PASS":
                return url
        except Exception:
            continue

    return candidates[0] if candidates else None


def validate_row_inputs(input_text: str, input_ocr: str, image_url: str | None) -> dict:
    checks = {}
    warnings = []
    image_payload = None

    checks["has_ocr"] = len(input_ocr) >= MIN_OCR_CHARS
    checks["has_input_text"] = bool(input_text)
    checks["has_image_url"] = bool(image_url)

    if not checks["has_ocr"] and not checks["has_image_url"]:
        return {
            "status": "FAIL",
            "notes": {
                "error": "No question source: OCR too short and image URL missing",
                "ocr_length": len(input_ocr),
                "image_url": image_url,
            },
            "image_payload": None,
            "user_prompt": "",
        }

    if image_url:
        if not is_valid_http_url(image_url):
            return {
                "status": "FAIL",
                "notes": {"error": f"Invalid image URL: {image_url}"},
                "image_payload": None,
                "user_prompt": build_user_prompt(input_text, input_ocr),
            }
        try:
            image_bytes, mime_type, status_code = download_image_bytes(image_url)
            image_payload = validate_image_bytes(image_bytes, mime_type)
            image_payload["notes"]["http_status"] = status_code
            image_payload["notes"]["image_url_host"] = urlparse(image_url).netloc
            if image_payload["status"] != "PASS":
                return {
                    "status": "FAIL",
                    "notes": image_payload["notes"],
                    "image_payload": None,
                    "user_prompt": build_user_prompt(input_text, input_ocr),
                }
        except Exception as exc:
            return {
                "status": "FAIL",
                "notes": {"error": f"Image download failed: {exc}", "image_url": image_url},
                "image_payload": None,
                "user_prompt": build_user_prompt(input_text, input_ocr),
            }
    else:
        warnings.append("No image URL; using OCR/text only")

    if not checks["has_ocr"] and image_payload:
        warnings.append(f"OCR short ({len(input_ocr)} chars); relying on image")

    status = "PASS"
    if warnings:
        status = "WARN"

    notes = {
        "checks": checks,
        "warnings": warnings,
        "ocr_length": len(input_ocr),
        "input_text_length": len(input_text),
    }
    if image_payload:
        notes["image"] = image_payload["notes"]

    return {
        "status": status,
        "notes": notes,
        "image_payload": image_payload,
        "user_prompt": build_user_prompt(input_text, input_ocr),
    }


def validate_model_output(response_text: str) -> dict:
    text = clean_text(response_text)
    checks = {}

    checks["nonempty"] = bool(text)
    checks["not_error_prefix"] = not text.upper().startswith("ERROR:")
    checks["min_length"] = len(text) >= MIN_SOLUTION_CHARS
    checks["has_final_answer"] = bool(
        re.search(r"final answer", text, re.IGNORECASE)
        or re.search(r"###\s*Final Answer", text, re.IGNORECASE)
    )
    checks["has_structure"] = bool(
        re.search(r"^#{1,3}\s+\w+", text, re.MULTILINE)
        or re.search(r"\bstep\s*\d+\b", text, re.IGNORECASE)
        or text.count("\n") >= 5
    )
    checks["has_math_or_reasoning"] = bool(
        re.search(r"\\\[.*?\\\]", text)
        or re.search(r"\\\(.*?\\\)", text)
        or re.search(r"\btherefore\b|\bbecause\b|\bhence\b", text, re.IGNORECASE)
    )

    hard_fail = not checks["nonempty"] or not checks["not_error_prefix"]
    status = "FAIL" if hard_fail else ("PASS" if all(checks.values()) else "WARN")

    return {
        "status": status,
        "notes": {
            "response_length": len(text),
            "checks": checks,
            "preview": text[:250],
        },
    }


def build_user_prompt(input_text: str, input_ocr: str) -> str:
    parts = []
    if input_ocr:
        parts.append(f"Question OCR:\n{input_ocr}")
    if input_text:
        parts.append(f"Additional input text:\n{input_text}")
    if not parts:
        return (
            "Solve the maths question shown in the image. "
            "Provide the complete step-by-step solution, not just the final answer."
        )
    parts.append(
        "Provide the complete step-by-step solution to this maths question. "
        "Include all important reasoning, equations, and a clearly labeled final answer."
    )
    return "\n\n".join(parts)


def prepare_contents(user_prompt: str, image_payload: dict | None = None):
    parts = []
    if user_prompt:
        parts.append(types.Part.from_text(text=user_prompt))
    if image_payload and image_payload.get("image_bytes"):
        parts.append(
            types.Part(
                inline_data=types.Blob(
                    data=image_payload["image_bytes"],
                    mime_type=image_payload["mime_type"],
                )
            )
        )
    if not parts:
        return []
    return [types.Content(role="user", parts=parts)]


def call_gemini(user_prompt: str, image_payload: dict | None = None) -> str:
    contents = prepare_contents(user_prompt, image_payload)
    if not contents:
        return ""

    config = types.GenerateContentConfig(
        system_instruction=[types.Part.from_text(text=SYSTEM_PROMPT)],
    )

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config,
            )
            return (response.text or "").strip()
        except Exception as exc:
            last_error = exc
            wait = attempt * 2
            print(f"Attempt {attempt} failed: {exc}. Retrying in {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"Gemini call failed after {MAX_RETRIES} attempts: {last_error}")


def notes_to_str(notes: dict) -> str:
    return json.dumps(notes, ensure_ascii=False)

In [5]:
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded {len(df)} rows")

# status/sub_status are required for filtering; other input cols are optional
required_cols = ["status", "sub_status"]
optional_cols = [
    "current_input_text",
    "current_input_image_url",
    "current_input_image_ocr",
]
COLUMN_ALIASES = {
    "current_input_text": ["user_input", "combined_input", "message"],
    "current_input_image_url": ["image_url", "static_image_url"],
    "current_input_image_ocr": ["image_ocr"],
}

missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

filled_from_alias = []
for col in optional_cols:
    if col not in df.columns:
        for alias in COLUMN_ALIASES.get(col, []):
            if alias in df.columns:
                df[col] = df[alias]
                filled_from_alias.append(f"{col} <- {alias}")
                break
    if col not in df.columns:
        df[col] = ""

if filled_from_alias:
    print("Mapped missing columns from aliases:")
    for mapping in filled_from_alias:
        print(f"  {mapping}")

still_missing = [c for c in optional_cols if c not in df.columns]
if still_missing:
    print(f"Optional columns not found (using empty defaults): {still_missing}")

# Only keep FRONTIER_MODEL_RESPONSE as the output column
drop_cols = [
    "frontier_input_status",
    "frontier_input_notes",
    "frontier_output_status",
    "frontier_output_notes",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

if "FRONTIER_MODEL_RESPONSE" not in df.columns:
    frontier_alias = next(
        (c for c in ["frontier_model_response", "FRONTIER_MODEL_RESPONSE"] if c in df.columns),
        None,
    )
    if frontier_alias:
        df["FRONTIER_MODEL_RESPONSE"] = df[frontier_alias].fillna("").astype(str)
    else:
        df["FRONTIER_MODEL_RESPONSE"] = ""
else:
    df["FRONTIER_MODEL_RESPONSE"] = df["FRONTIER_MODEL_RESPONSE"].fillna("").astype(str)

# Prefer static_image_url when current_input_image_url is missing or presigned/expired
if "static_image_url" in df.columns:
    static = df["static_image_url"].fillna("").astype(str).str.strip()
    current = df["current_input_image_url"].fillna("").astype(str).str.strip()
    use_static = static.ne("") & (
        current.eq("")
        | current.str.contains("prod-doubts", case=False, na=False)
        | current.str.contains("X-Amz-", na=False)
    )
    if use_static.any():
        df.loc[use_static, "current_input_image_url"] = static[use_static]
        print(f"Rewrote current_input_image_url from static_image_url for {use_static.sum()} rows")


def is_target_row(row) -> bool:
    return (
        clean_text(row.get("status")) == TARGET_STATUS
        and clean_text(row.get("sub_status")) == TARGET_SUB_STATUS
    )

target_mask = df.apply(is_target_row, axis=1)
target_df = df[target_mask].copy()

print("\nRow breakdown:")
print(df.groupby(["status", "sub_status"]).size().to_string())
print(f"\nTarget rows (NO_ANSWER + VERIFIER_REJECTED): {len(target_df)}")
print(f"Skipped rows (SUCCESS etc.): {len(df) - len(target_df)}")

preview_cols = [
    c
    for c in [
        "status",
        "sub_status",
        "current_input_text",
        "current_input_image_url",
        "current_input_image_ocr",
        "FRONTIER_MODEL_RESPONSE",
    ]
    if c in target_df.columns
]
target_df.head(3)[preview_cols]

Loaded 19 rows

Row breakdown:
status     sub_status       
NO_ANSWER  VERIFIER_REJECTED    19

Target rows (NO_ANSWER + VERIFIER_REJECTED): 19
Skipped rows (SUCCESS etc.): 0


,status,sub_status,current_input_text,current_input_image_url,current_input_image_ocr,FRONTIER_MODEL_RESPONSE
0,NO_ANSWER,VERIFIER_REJECTED,H\n\nA rod of length a is broken into three pa...,https://doubt-qc.s3.ap-south-1.amazonaws.com/d...,H\n\nA rod of length a is broken into three pa...,
1,NO_ANSWER,VERIFIER_REJECTED,3. (i) If the set of values of \( x \) satisfy...,https://doubt-qc.s3.ap-south-1.amazonaws.com/d...,3. (i) If the set of values of \( x \) satisfy...,
2,NO_ANSWER,VERIFIER_REJECTED,For what values of a do the graphs of the func...,https://doubt-qc.s3.ap-south-1.amazonaws.com/d...,For what values of a do the graphs of the func...,


In [6]:
# Pre-flight: validate inputs for TARGET rows only (console only, not saved)

preflight_rows = []
for idx, row in target_df.iterrows():
    input_text = clean_text(row.get("current_input_text"))
    input_ocr = clean_text(row.get("current_input_image_ocr"))
    image_url = resolve_image_url(row)
    result = validate_row_inputs(input_text, input_ocr, image_url)
    preflight_rows.append({
        "row": idx,
        "message_id": row.get("message_id", ""),
        "input_status": result["status"],
    })

preflight_df = pd.DataFrame(preflight_rows)
print(f"Pre-flight on {len(preflight_df)} verifier-rejected rows:")
print(preflight_df["input_status"].value_counts(dropna=False).to_string())
print(f"\nInput FAIL rows: {(preflight_df['input_status'] == 'FAIL').sum()}")

Pre-flight on 19 verifier-rejected rows:
input_status
PASS    19

Input FAIL rows: 0


In [7]:
def row_done(value: str) -> bool:
    text = clean_text(value)
    return bool(text) and not text.upper().startswith("ERROR:")


from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

save_lock = threading.Lock()
completed_count = 0


def process_one_row(idx: int) -> tuple[int, str | None, bool]:
    """Returns (idx, answer_or_none, input_failed)."""
    row = df.loc[idx]
    input_text = clean_text(row.get("current_input_text"))
    input_ocr = clean_text(row.get("current_input_image_ocr"))
    image_url = resolve_image_url(row)

    input_result = validate_row_inputs(input_text, input_ocr, image_url)
    if input_result["status"] == "FAIL":
        return idx, None, True

    try:
        answer = call_gemini(
            user_prompt=input_result["user_prompt"],
            image_payload=input_result["image_payload"],
        )
    except Exception as exc:
        answer = f"ERROR: {exc}"

    return idx, answer, False


pending_idx = []
for idx, row in target_df.iterrows():
    if SKIP_COMPLETED and row_done(row.get("FRONTIER_MODEL_RESPONSE", "")):
        continue
    pending_idx.append(idx)

print(f"Verifier-rejected rows to process: {len(pending_idx)} / {len(target_df)}")
print(f"SUCCESS rows skipped entirely: {len(df) - len(target_df)}")
print(f"Using {MAX_WORKERS} parallel workers")

skipped_input_fail = 0
workers = max(1, min(MAX_WORKERS, len(pending_idx) or 1))

with ThreadPoolExecutor(max_workers=workers) as executor:
    futures = {executor.submit(process_one_row, idx): idx for idx in pending_idx}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Gemini Pro"):
        idx, answer, input_failed = future.result()

        if input_failed:
            skipped_input_fail += 1
            print(f"Skipping row {idx}: input validation failed")
            continue

        with save_lock:
            df.at[idx, "FRONTIER_MODEL_RESPONSE"] = answer
            completed_count += 1
            if completed_count % SAVE_EVERY == 0 or completed_count == len(pending_idx):
                df.to_csv(CSV_PATH, index=False)
                print(f"Checkpoint saved after {completed_count} processed rows -> {CSV_PATH}")

df.to_csv(CSV_PATH, index=False)
filled_target = sum(
    bool(clean_text(df.at[i, "FRONTIER_MODEL_RESPONSE"])) for i in target_df.index
)
print("Done.")
print(f"Filled FRONTIER_MODEL_RESPONSE for {filled_target} / {len(target_df)} verifier-rejected rows")
print(f"Skipped due to input validation fail: {skipped_input_fail}")

Verifier-rejected rows to process: 19 / 19
SUCCESS rows skipped entirely: 0
Using 5 parallel workers


Gemini Pro:   0%|          | 0/19 [00:00<?, ?it/s]

Gemini Pro:  26%|██▋       | 5/19 [01:06<02:45, 11.81s/it]

Checkpoint saved after 5 processed rows -> /Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/3_rej_jee_math_truncated/eval_results_unified_145_merged_19_rejected.csv


Gemini Pro:  53%|█████▎    | 10/19 [02:20<01:52, 12.55s/it]

Checkpoint saved after 10 processed rows -> /Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/3_rej_jee_math_truncated/eval_results_unified_145_merged_19_rejected.csv


Gemini Pro:  79%|███████▉  | 15/19 [03:08<00:42, 10.64s/it]

Checkpoint saved after 15 processed rows -> /Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/3_rej_jee_math_truncated/eval_results_unified_145_merged_19_rejected.csv


Gemini Pro: 100%|██████████| 19/19 [04:30<00:00, 14.23s/it]

Checkpoint saved after 19 processed rows -> /Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/3_rej_jee_math_truncated/eval_results_unified_145_merged_19_rejected.csv
Done.
Filled FRONTIER_MODEL_RESPONSE for 19 / 19 verifier-rejected rows
Skipped due to input validation fail: 0


In [8]:
# Post-run preview (verifier-rejected rows only)

results_df = df[target_mask].copy()
filled = results_df["FRONTIER_MODEL_RESPONSE"].astype(str).str.strip().ne("").sum()

print(f"Verifier-rejected rows: {len(results_df)}")
print(f"Rows with FRONTIER_MODEL_RESPONSE filled: {filled}")

preview = results_df[
    [
        "message_id",
        "status",
        "sub_status",
        "current_input_image_ocr",
        "FRONTIER_MODEL_RESPONSE",
    ]
].copy()
preview["FRONTIER_MODEL_RESPONSE"] = preview["FRONTIER_MODEL_RESPONSE"].astype(str).str.slice(0, 300)
preview.head(10)

Verifier-rejected rows: 19
Rows with FRONTIER_MODEL_RESPONSE filled: 19


,message_id,status,sub_status,current_input_image_ocr,FRONTIER_MODEL_RESPONSE
0,77362aae-6c28-4094-ade4-5e241d90e122,NO_ANSWER,VERIFIER_REJECTED,H\n\nA rod of length a is broken into three pa...,"### Introduction\nTo solve this problem, we ca..."
1,8b36115a-6be0-4518-b676-64ff33e1eb8b,NO_ANSWER,VERIFIER_REJECTED,3. (i) If the set of values of \( x \) satisfy...,### Introduction\n\nTo solve the given logarit...
2,712d5dd9-5bb5-4797-870e-3112213193c9,NO_ANSWER,VERIFIER_REJECTED,For what values of a do the graphs of the func...,### Introduction\nThe problems presented invol...
3,ebf79bf4-fcfd-4d71-bc9f-5aa311510617,NO_ANSWER,VERIFIER_REJECTED,5. For what values of a do the graphs of the f...,### Introduction\n\nTo find the values of $a$ ...
4,96023252-7172-4268-a736-4166d1ce5003,NO_ANSWER,VERIFIER_REJECTED,(iii) \( \frac{x^{2}+2}{x^{2}-1}<-2 \),### Introduction\nTo solve the given rational ...
5,a4981214-4881-4654-86f6-a61bda5a37ad,NO_ANSWER,VERIFIER_REJECTED,smallest integral good number.\n\nQEANG\n\nNum...,Here are the step-by-step solutions for the tw...
6,fa33f671-2495-422a-ab45-3c63a743bf8f,NO_ANSWER,VERIFIER_REJECTED,19. Find set of values of ' \( a \) ' for whic...,Here are the complete step-by-step solutions f...
7,bd5f8be3-3a23-4ef5-8f9d-50b49e13106b,NO_ANSWER,VERIFIER_REJECTED,(D) None of these\n10. Equation \( 2 x^{2}-2(2...,Here are the step-by-step solutions for the qu...
8,671a8ed1-63b4-463d-bbcf-3215d9a3cae4,NO_ANSWER,VERIFIER_REJECTED,(xi) \( \frac{\mathrm{x}^{2}+2}{\mathrm{x}^{2}...,### Introduction\nTo solve the given rational ...
9,19f61901-e2ee-446b-ba8b-22b2c3804c6f,NO_ANSWER,VERIFIER_REJECTED,"For \( 0<c<b<a \), let \( (a+b-2 c) x^{2}+(b+c...","Based on the image and text provided, there ar..."


In [10]:
from pathlib import Path

INPUT_CSV = Path("/Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/3_rej_jee_math_truncated/eval_results_unified_145_merged_19_rejected.csv")
OUTPUT_CSV = INPUT_CSV.with_name(INPUT_CSV.stem + "_verifier_rejected.csv")

df_all = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(df_all)} rows from {INPUT_CSV.name}")

rejected_df = df_all[
    df_all["sub_status"].astype(str).str.strip().str.upper() == "VERIFIER_REJECTED"
].copy()

print("\nsub_status breakdown (all rows):")
print(df_all["sub_status"].value_counts(dropna=False).to_string())

print(f"\nFiltered VERIFIER_REJECTED rows: {len(rejected_df)}")
rejected_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved -> {OUTPUT_CSV}")

rejected_df.head(3)[
    [c for c in ["status", "sub_status", "current_input_text", "current_input_image_url", "current_input_image_ocr", "verifier_response"] if c in rejected_df.columns]
]

Loaded 19 rows from eval_results_unified_145_merged_19_rejected.csv

sub_status breakdown (all rows):
sub_status
VERIFIER_REJECTED    19

Filtered VERIFIER_REJECTED rows: 19
Saved -> /Users/simrannaik/Desktop/bots/neet_bots/bot-verifier-script/3_rej_jee_math_truncated/eval_results_unified_145_merged_19_rejected_verifier_rejected.csv


,status,sub_status,current_input_text,current_input_image_url,current_input_image_ocr,verifier_response
0,NO_ANSWER,VERIFIER_REJECTED,H\n\nA rod of length a is broken into three pa...,https://doubt-qc.s3.ap-south-1.amazonaws.com/d...,H\n\nA rod of length a is broken into three pa...,NaN
1,NO_ANSWER,VERIFIER_REJECTED,3. (i) If the set of values of \( x \) satisfy...,https://doubt-qc.s3.ap-south-1.amazonaws.com/d...,3. (i) If the set of values of \( x \) satisfy...,NaN
2,NO_ANSWER,VERIFIER_REJECTED,For what values of a do the graphs of the func...,https://doubt-qc.s3.ap-south-1.amazonaws.com/d...,For what values of a do the graphs of the func...,NaN
